# daftar in a notebook

Notebooks are where provenance dies. The git commit is close to meaningless —
a notebook is one file whose cells ran in an order nobody recorded. Cells get
edited and re-run, so the code that made a figure may no longer exist anywhere.
And on Colab there is no repository at all.

daftar records the two things that actually identify a notebook result:

- **`code.cell_sha256`** — the source of the cell that ran, captured *before*
  execution, so it survives you editing the cell afterwards.
- **`code.session_history_sha256`** — every cell executed before it. A result
  depends on the whole session, and nothing else records that.

Run the cells in order, then run the **out-of-order** section at the end.

In [ ]:
!pip install -q daftar

In [ ]:
%load_ext daftar
import daftar
print("daftar", daftar.__version__)

## 1. A tracked cell

`%%daftar <label> seed=<n>` wraps the cell. A `run` object is injected — no import needed.

In [ ]:
import numpy as np
scale = 1.0

In [ ]:
%%daftar montecarlo seed=42
rng = np.random.default_rng(0)
vals = rng.normal(size=100_000) * scale
run.log_result("mean", float(vals.mean()))
run.log_result("std", float(vals.std()))
print("mean", vals.mean())

## 2. What was recorded

In [ ]:
store = daftar.RunStore()
m = store.load(store.list()[0].run_id)
for k in m.ordered_keys:
    if k.split(".")[0] in ("code", "seed", "result"):
        print(f"{k:<34} {m.fields[k][:60]}")

## 3. The thing no other tool catches

Change a variable an earlier cell defined, then re-run **the identical cell**.
Same code, same seed, same everything on disk — different answer.

In [ ]:
scale = 3.0   # an upstream cell changes

In [ ]:
%%daftar montecarlo seed=42
rng = np.random.default_rng(0)
vals = rng.normal(size=100_000) * scale
run.log_result("mean", float(vals.mean()))
run.log_result("std", float(vals.std()))
print("mean", vals.mean())

In [ ]:
runs = daftar.RunStore().list(limit=2)
a, b = store.load(runs[1].run_id), store.load(runs[0].run_id)
print(daftar.render_diff(daftar.diff_manifests(a, b)))

The cell hash is **identical** — it is the same code. What differs is
`code.session_history_sha256`: the session state that produced the number.

Without that field this would have been reported as *nondeterministic*, which
would have been wrong and would have sent you looking for a seeding bug that
does not exist.

## 4. Without the magic

`daftar.track()` detects the notebook by itself. Existing code needs no changes
— it simply gains the notebook fields.

In [ ]:
with daftar.track("plain-track", params={"n": 1000}, seed=7) as run:
    x = np.random.default_rng(run.seed).normal(size=1000)
    run.log_result("mean", float(x.mean()))

m = store.load(run.run_id)
print("entrypoint       ", m.get("code.entrypoint"))
print("cell hash        ", m.get("code.cell_sha256"))
print("session history  ", m.get("code.session_history_sha256"))
print("cells before this", m.get("code.session_n_cells"))

## 5. Hand it to someone else

The bundle contains `cell.py` — the source that actually ran. On Colab, where
the VM is ephemeral and nothing is committed, this is often the only surviving
copy of the code that produced the result.

In [ ]:
bundle = daftar.export_bundle(m, "run.zip", store=store)
import zipfile
print(zipfile.ZipFile(bundle).namelist())
print(daftar.plan_replay(m, check_current=False).render())

## 6. Try it on your own work

Add one line to a notebook you already have:

```python
with daftar.track("my-experiment", seed=42) as run:
    ...            # your existing code, unchanged
    run.log_result("accuracy", acc)
```

Then run it twice on different days and `daftar diff` the two.